In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
!pip install -q qdrant-client
!pip install -q langchain
!pip install -q langchain-openai
!pip install -q langchain-naver
!pip install -q rank_bm25

In [ ]:
# 모듈 불러오기
%run /content/drive/MyDrive/aiffel_final_pjt/src/state/state_v1.ipynb
%run /content/drive/MyDrive/aiffel_final_pjt/src/db/qdrant.py
%run /content/drive/MyDrive/aiffel_final_pjt/src/embedding/embedder.py

In [ ]:
import os
from google.colab import userdata

# Set the OpenAI API key environment variable
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

# Set the HCX API key environment variable
os.environ["CLOVASTUDIO_API_KEY"] = userdata.get('CLOVASTUDIO_API_KEY')

# Set the Qdrant API key environment variable
os.environ["QDRANT_API_KEY"] = userdata.get('QDRANT_API_KEY')
os.environ["QDRANT_URL"] = userdata.get('QDRANT_URL')

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_naver import ChatClovaX

# ── 초기화 ──────────────────────────────
embedder = LocalEmbedder("BAAI/bge-m3")   # 임베딩
db = QdrantDB(vector_size=1024)           # Qdrant 연결

# LLM
llm = ChatOpenAI(model="gpt-4o-mini")     # OpenAI LLM
# llm = ChatClovaX(model="HCX-005", temperature=0)

## Fusion RAG Node

**simple_rag_node** 대비 변경 사항:
- Qdrant에서 후보 풀(`POOL_SIZE`)을 넉넉하게 뽑은 뒤
- 그 후보들의 `book_intro`로 BM25 인덱스를 즉석 생성
- Vector 점수와 BM25 점수를 각각 Min-Max 정규화 후 가중 합산(`alpha`)
- 최종 상위 `TOP_K`개를 `retrieved_books`로 반환

| 하이퍼파라미터 | 기본값 | 설명 |
|---|---|---|
| `POOL_SIZE` | 20 | BM25 재정렬 대상 후보 수 |
| `TOP_K` | 5 | 최종 반환 도서 수 |
| `ALPHA` | 0.5 | Vector 비중 (1.0=Vector only, 0.0=BM25 only) |

In [ ]:
from langgraph.graph import StateGraph, START, END
from langchain_core.messages import HumanMessage, AIMessage
from rank_bm25 import BM25Okapi
import numpy as np
import json


# ── 하이퍼파라미터 ───────────────────────
POOL_SIZE = 20   # BM25 재정렬 대상 후보 수
TOP_K     = 5    # 최종 반환 도서 수
ALPHA     = 0.5  # Vector 비중 (0.0 ~ 1.0)


def fusion_rag_node(state: CRSState) -> dict:
    summary = state.get("summary", "")

    # ── 1. Vector 검색: 후보 풀 확보 ────────────────────────────
    query_vector = embedder.embed(summary)
    vector_results = db.search("books_v1", query_vector, limit=POOL_SIZE, threshold=0.3)

    if not vector_results:
        return {"retrieved_books": []}

    # ── 2. BM25 인덱스 생성 (후보 풀의 book_intro 대상) ─────────
    corpus_texts  = [r.payload.get("book_intro", "") for r in vector_results]
    tokenized_corpus = [text.split() for text in corpus_texts]
    bm25 = BM25Okapi(tokenized_corpus)

    # ── 3. BM25 점수 계산 ────────────────────────────────────────
    tokenized_query = summary.split()
    bm25_scores = bm25.get_scores(tokenized_query)           # shape: (POOL_SIZE,)

    # ── 4. Vector 점수 추출 ──────────────────────────────────────
    vector_scores = np.array([r.score for r in vector_results])  # cosine similarity

    # ── 5. Min-Max 정규화 ────────────────────────────────────────
    def normalize(scores: np.ndarray) -> np.ndarray:
        min_s, max_s = scores.min(), scores.max()
        if max_s - min_s < 1e-8:
            return np.zeros_like(scores)
        return (scores - min_s) / (max_s - min_s)

    norm_vector = normalize(vector_scores)
    norm_bm25   = normalize(bm25_scores)

    # ── 6. Fusion 점수 합산 ──────────────────────────────────────
    combined = ALPHA * norm_vector + (1 - ALPHA) * norm_bm25

    # ── 7. 상위 TOP_K 선택 ───────────────────────────────────────
    top_indices = np.argsort(combined)[::-1][:TOP_K]

    retrieved_books = [
        {
            "isbn":      vector_results[i].payload.get("isbn"),
            "title":     vector_results[i].payload.get("title"),
            "author":    vector_results[i].payload.get("author"),
            "book_intro": vector_results[i].payload.get("book_intro"),
        }
        for i in top_indices
    ]

    return {"retrieved_books": retrieved_books}

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

def rag_llm_node(state: CRSState) -> dict:
    summary = state.get("summary", "")
    books = state["retrieved_books"]

    context = "\n\n".join([
        f"ISBN: {b['isbn']}\n"
        f"제목: {b['title']}\n"
        f"저자: {b['author']}\n"
        f"소개: {b['book_intro']}"
        for b in books
    ])

    # 프롬프트 직접 정의
    rag_prompt = ChatPromptTemplate.from_template("""
당신은 도서관 큐레이터 AI입니다.

[규칙]
- 반드시 [검색된 도서 목록]에 있는 책만 추천하세요.
- 반드시 JSON 형식으로만 답하세요. 다른 텍스트는 절대 포함하지 마세요.
- 사용자 프로파일을 참고해서 가장 적합한 도서 3권을 추천하세요.
- 추천 이유는 반드시 사용자 프로파일의 독서 목적, 성향, 상황과 연결해서 작성하세요.

[사용자 프로파일]
{summary}

[검색된 도서 목록]
{context}

[출력 형식]
[
    {{"title": "책 제목", "author": "저자", "isbn": "ISBN번호", "reason": "추천 이유 2~3문장"}},
    {{"title": "책 제목", "author": "저자", "isbn": "ISBN번호", "reason": "추천 이유 2~3문장"}},
    {{"title": "책 제목", "author": "저자", "isbn": "ISBN번호", "reason": "추천 이유 2~3문장"}}
]
"""
    )

    chain = rag_prompt | llm
    response = chain.invoke({
        "context": context,
        "summary": summary
    })

    try:
        recommendations = json.loads(response.content)
    except json.JSONDecodeError:
        # 파싱 실패해도 LLM 응답 그대로 반환
        recommendations = response.content

    # recommendations 출력 확인
    # print(recommendations)

    return {
        "messages": [AIMessage(content=response.content)],
        "recommendations": recommendations
    }

In [ ]:
# # ── TEST 실행 ────────────────────────────────
# graph = StateGraph(CRSState)
# graph.add_node("rag", fusion_rag_node)  # simple_rag_node → fusion_rag_node
# graph.add_node("llm", rag_llm_node)
# graph.add_edge(START, "rag")
# graph.add_edge("rag", "llm")
# graph.add_edge("llm", END)
# app = graph.compile()

# result = app.invoke({
#     "messages": [HumanMessage(content="번아웃이 심한데 위로가 되는 소설 추천해줘")],
#     "retrieved_books": [],
#     "recommendations": []
# })

# print(result["messages"][-1].content)

In [ ]:
# result["recommendations"]